<a href="https://colab.research.google.com/github/f2023266907-hiba/hello-world/blob/main/Ai_Exam_Scheduler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AI EXAM SCHEDULER.


HIBA AAMIR

F20232907

SECTION V2


In [1]:
from typing import List, Dict, Tuple, Optional, Set
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import random
import json
from collections import defaultdict
import sys


check if code is runnng in colab:

In [2]:
try:
    import google.colab
    IN_COLAB = True
    print("✅ Running in Google Colab")
except:
    IN_COLAB = False
    print("✅ Running locally")

✅ Running in Google Colab


In [3]:
@dataclass
class Course:
    """Represents a course that needs an exam"""
    id: str
    name: str
    instructor: str
    students: List[str]
    priority: int = 3
    duration: int = 120
    preferred_times: Optional[List[str]] = None
    prerequisites: Optional[List[str]] = None

@dataclass
class ExamSlot:
    """Represents an available exam time slot"""
    id: str
    date: str
    start_time: str
    end_time: str
    capacity: int = 50
    venue: str = ""

@dataclass
class ExamSchedule:
    """Represents a scheduled exam"""
    course_id: str
    slot_id: str
    date: str
    start_time: str
    end_time: str
    venue: str
    priority: int

 EXAM SCHEDULER CLASS

In [4]:
class ExamScheduler:
    """AI Exam Scheduler using CSP with backtracking and priority constraints"""

    def __init__(self):
        self.courses: List[Course] = []
        self.slots: List[ExamSlot] = []
        self.schedule: List[ExamSchedule] = []
        self.constraints: List[Dict] = []
        self.backtrack_count = 0

    def add_course(self, course: Course):
        if not 1 <= course.priority <= 5:
            raise ValueError("Priority must be between 1 and 5")
        self.courses.append(course)
        print(f"✅ Added course: {course.name} (ID: {course.id}, Priority: {course.priority})")

    def add_slot(self, slot: ExamSlot):
        self.slots.append(slot)
        print(f"✅ Added slot: {slot.date} {slot.start_time}-{slot.end_time} at {slot.venue}")

    def add_constraint(self, constraint_type: str, course_ids: List[str],
                      constraint_value: any = None):
        self.constraints.append({
            'type': constraint_type,
            'courses': course_ids,
            'value': constraint_value
        })
        print(f"🔒 Added constraint: {constraint_type} for {', '.join(course_ids)}")

    def get_course_students(self, course_id: str) -> Set[str]:
        for course in self.courses:
            if course.id == course_id:
                return set(course.students)
        return set()

    def check_student_conflict(self, course1_id: str, course2_id: str) -> bool:
        students1 = self.get_course_students(course1_id)
        students2 = self.get_course_students(course2_id)
        return bool(students1 & students2)

    def check_time_overlap(self, slot1: ExamSlot, slot2: ExamSlot) -> bool:
        start1 = datetime.strptime(f"{slot1.date} {slot1.start_time}", "%Y-%m-%d %H:%M")
        end1 = datetime.strptime(f"{slot1.date} {slot1.end_time}", "%Y-%m-%d %H:%M")
        start2 = datetime.strptime(f"{slot2.date} {slot2.start_time}", "%Y-%m-%d %H:%M")
        end2 = datetime.strptime(f"{slot2.date} {slot2.end_time}", "%Y-%m-%d %H:%M")
        return not (end1 <= start2 or end2 <= start1)

    def get_time_diff(self, slot1: ExamSlot, slot2: ExamSlot) -> float:
        start1 = datetime.strptime(f"{slot1.date} {slot1.start_time}", "%Y-%m-%d %H:%M")
        start2 = datetime.strptime(f"{slot2.date} {slot2.start_time}", "%Y-%m-%d %H:%M")
        return abs((start2 - start1).total_seconds() / 3600)

    def get_slot_by_id(self, slot_id: str) -> Optional[ExamSlot]:
        for slot in self.slots:
            if slot.id == slot_id:
                return slot
        return None

    def is_valid_assignment(self, course: Course, slot: ExamSlot,
                           current_schedule: List[ExamSchedule]) -> bool:
        # Check capacity
        scheduled_count = sum(1 for s in current_schedule if s.slot_id == slot.id)
        if scheduled_count >= slot.capacity:
            return False

        # Check student conflicts
        for scheduled in current_schedule:
            if self.check_student_conflict(course.id, scheduled.course_id):
                scheduled_slot = self.get_slot_by_id(scheduled.slot_id)
                if scheduled_slot:
                    if self.get_time_diff(slot, scheduled_slot) < 2:
                        return False

        # Check custom constraints
        for constraint in self.constraints:
            if course.id in constraint['courses']:
                if constraint['type'] == 'same_day':
                    for scheduled in current_schedule:
                        if scheduled.course_id in constraint['courses']:
                            if scheduled.date != slot.date:
                                return False

                elif constraint['type'] == 'different_day':
                    for scheduled in current_schedule:
                        if scheduled.course_id in constraint['courses']:
                            if scheduled.date == slot.date:
                                return False

                elif constraint['type'] == 'no_back_to_back':
                    for scheduled in current_schedule:
                        if scheduled.course_id in constraint['courses']:
                            scheduled_slot = self.get_slot_by_id(scheduled.slot_id)
                            if scheduled_slot:
                                if self.get_time_diff(slot, scheduled_slot) < 2:
                                    return False

                elif constraint['type'] == 'priority_time':
                    if course.preferred_times:
                        if slot.start_time not in course.preferred_times:
                            return False

        return True

    def calculate_slot_score(self, course: Course, slot: ExamSlot,
                            current_schedule: List[ExamSchedule]) -> float:
        score = 0.0
        score += course.priority * 10

        if course.preferred_times and slot.start_time in course.preferred_times:
            score += 20

        conflict_count = 0
        for scheduled in current_schedule:
            if self.check_student_conflict(course.id, scheduled.course_id):
                conflict_count += 1
        score -= conflict_count * 5

        hour = int(slot.start_time.split(':')[0])
        if 9 <= hour <= 11:
            score += 15
        elif 12 <= hour <= 14:
            score += 10
        elif 15 <= hour <= 17:
            score += 5

        scheduled_count = sum(1 for s in current_schedule if s.slot_id == slot.id)
        free_capacity = slot.capacity - scheduled_count
        score += min(free_capacity, 10)

        return score

    def solve_csp(self, use_priority: bool = True, max_attempts: int = 1000) -> List[ExamSchedule]:
        print("\n" + "="*60)
        print("🤖 AI Exam Scheduler - Solving CSP")
        print("="*60)

        self.backtrack_count = 0

        if use_priority:
            sorted_courses = sorted(self.courses,
                                  key=lambda c: (-c.priority, -len(c.students)))
        else:
            sorted_courses = sorted(self.courses, key=lambda c: -len(c.students))

        schedule = []
        assigned = set()

        for course in sorted_courses:
            valid_slots = []
            for slot in self.slots:
                if self.is_valid_assignment(course, slot, schedule):
                    valid_slots.append(slot)

            if not valid_slots:
                print(f"⚠️ No valid slots found for {course.name} (ID: {course.id})")
                continue

            scored_slots = []
            for slot in valid_slots:
                score = self.calculate_slot_score(course, slot, schedule)
                scored_slots.append((score, slot))

            scored_slots.sort(key=lambda x: x[0], reverse=True)
            best_score, best_slot = scored_slots[0]

            schedule.append(ExamSchedule(
                course_id=course.id,
                slot_id=best_slot.id,
                date=best_slot.date,
                start_time=best_slot.start_time,
                end_time=best_slot.end_time,
                venue=best_slot.venue,
                priority=course.priority
            ))
            assigned.add(course.id)

            print(f"📚 Scheduled {course.name} (Priority: {course.priority}) -> "
                  f"{best_slot.date} {best_slot.start_time}-{best_slot.end_time} "
                  f"at {best_slot.venue} (Score: {best_score:.1f})")

        unscheduled = [c for c in self.courses if c.id not in assigned]
        if unscheduled:
            print(f"\n🔄 Attempting to resolve {len(unscheduled)} unscheduled courses...")
            self.backtrack_solve(unscheduled, schedule, 0, max_attempts)

        self.schedule = schedule
        print(f"\n✅ Scheduling complete! Scheduled {len(schedule)}/{len(self.courses)} courses")
        print(f"   Backtracking attempts: {self.backtrack_count}")

        return schedule

    def backtrack_solve(self, courses: List[Course], current_schedule: List[ExamSchedule],
                       index: int, max_attempts: int) -> bool:
        self.backtrack_count += 1

        if self.backtrack_count > max_attempts:
            return False

        if index >= len(courses):
            return True

        course = courses[index]

        for slot in self.slots:
            if self.is_valid_assignment(course, slot, current_schedule):
                current_schedule.append(ExamSchedule(
                    course_id=course.id,
                    slot_id=slot.id,
                    date=slot.date,
                    start_time=slot.start_time,
                    end_time=slot.end_time,
                    venue=slot.venue,
                    priority=course.priority
                ))

                if self.backtrack_solve(courses, current_schedule, index + 1, max_attempts):
                    return True

                current_schedule.pop()

        return False

    def get_statistics(self) -> Dict:
        stats = {
            'total_courses': len(self.courses),
            'total_slots': len(self.slots),
            'scheduled_courses': len(self.schedule),
            'unscheduled_courses': len(self.courses) - len(self.schedule),
            'backtracking_attempts': self.backtrack_count,
            'scheduling_rate': len(self.schedule) / len(self.courses) if self.courses else 0,
            'priority_breakdown': {},
            'venue_usage': {},
            'daily_distribution': {}
        }

        for course in self.courses:
            priority = course.priority
            if priority not in stats['priority_breakdown']:
                stats['priority_breakdown'][priority] = {'total': 0, 'scheduled': 0}
            stats['priority_breakdown'][priority]['total'] += 1
            if any(s.course_id == course.id for s in self.schedule):
                stats['priority_breakdown'][priority]['scheduled'] += 1

        for sched in self.schedule:
            venue = sched.venue
            stats['venue_usage'][venue] = stats['venue_usage'].get(venue, 0) + 1

        for sched in self.schedule:
            date = sched.date
            stats['daily_distribution'][date] = stats['daily_distribution'].get(date, 0) + 1

        return stats

    def print_statistics(self):
        stats = self.get_statistics()

        print("\n" + "="*60)
        print("📊 SCHEDULING STATISTICS")
        print("="*60)
        print(f"Total Courses: {stats['total_courses']}")
        print(f"Total Slots: {stats['total_slots']}")
        print(f"Scheduled Courses: {stats['scheduled_courses']}")
        print(f"Unscheduled Courses: {stats['unscheduled_courses']}")
        print(f"Scheduling Rate: {stats['scheduling_rate']*100:.1f}%")
        print(f"Backtracking Attempts: {stats['backtracking_attempts']}")

        print("\n📈 Priority Breakdown:")
        for priority in sorted(stats['priority_breakdown'].keys(), reverse=True):
            data = stats['priority_breakdown'][priority]
            rate = data['scheduled'] / data['total'] * 100 if data['total'] > 0 else 0
            print(f"  Priority {priority}: {data['scheduled']}/{data['total']} ({rate:.1f}%)")

        print("\n🏛️ Venue Usage:")
        for venue, count in sorted(stats['venue_usage'].items(), key=lambda x: x[1], reverse=True):
            print(f"  {venue}: {count} exams")

        print("\n📅 Daily Distribution:")
        for date, count in sorted(stats['daily_distribution'].items()):
            print(f"  {date}: {count} exams")

    def print_schedule(self):
        if not self.schedule:
            print("No schedule generated yet.")
            return

        print("\n" + "="*60)
        print("📅 EXAM SCHEDULE")
        print("="*60)

        by_date = defaultdict(list)
        for sched in self.schedule:
            by_date[sched.date].append(sched)

        for date in sorted(by_date.keys()):
            print(f"\n📆 {date}")
            print("-"*50)
            for sched in sorted(by_date[date], key=lambda x: x.start_time):
                course = next((c for c in self.courses if c.id == sched.course_id), None)
                name = course.name if course else sched.course_id
                priority_stars = "⭐" * sched.priority
                print(f"  {sched.start_time}-{sched.end_time} | {name} | "
                      f"{sched.venue} | {priority_stars}")

    def save_schedule_json(self, filename: str = "exam_schedule.json"):
        schedule_data = []
        for item in self.schedule:
            course = next((c for c in self.courses if c.id == item.course_id), None)
            if course:
                schedule_data.append({
                    'course_id': item.course_id,
                    'course_name': course.name,
                    'instructor': course.instructor,
                    'date': item.date,
                    'start_time': item.start_time,
                    'end_time': item.end_time,
                    'venue': item.venue,
                    'priority': item.priority,
                    'student_count': len(course.students),
                    'students': course.students
                })

        with open(filename, 'w') as f:
            json.dump(schedule_data, f, indent=2)

        print(f"💾 Schedule saved to {filename}")

        # In Colab, download the file automatically
        if IN_COLAB:
            try:
                from google.colab import files
                files.download(filename)
                print("📥 File downloaded automatically!")
            except:
                pass

# DEMO FUNCTION

def run_demo():
    """Run the complete demo in Colab"""

    print("\n" + "🚀"*20)
    print("AI EXAM SCHEDULER - COLAB DEMO")
    print("🚀"*20 + "\n")

    # Create scheduler

    scheduler = ExamScheduler()


    # 1. ADD COURSES

    print("\n📚 Adding Courses...")
    print("-"*40)

    courses = [
        Course(
            id="CS101",
            name="Data Structures",
            instructor="Dr. Smith",
            students=["S001", "S002", "S003", "S004", "S005"],
            priority=5,
            preferred_times=["09:00", "10:00"]
        ),
        Course(
            id="CS102",
            name="Algorithms",
            instructor="Dr. Johnson",
            students=["S001", "S003", "S006", "S007"],
            priority=4,
            preferred_times=["10:00"]
        ),
        Course(
            id="CS103",
            name="Database Systems",
            instructor="Prof. Williams",
            students=["S002", "S004", "S006", "S008", "S009"],
            priority=4
        ),
        Course(
            id="CS104",
            name="Operating Systems",
            instructor="Dr. Brown",
            students=["S001", "S005", "S007", "S010"],
            priority=5,
            preferred_times=["09:00"]
        ),
        Course(
            id="CS105",
            name="Computer Networks",
            instructor="Prof. Davis",
            students=["S003", "S008", "S009", "S010"],
            priority=3
        ),
        Course(
            id="CS106",
            name="Software Engineering",
            instructor="Dr. Wilson",
            students=["S002", "S004", "S005", "S006", "S007"],
            priority=3
        ),
        Course(
            id="CS107",
            name="Artificial Intelligence",
            instructor="Prof. Taylor",
            students=["S001", "S009", "S010"],
            priority=5,
            preferred_times=["09:00", "11:00"]
        ),
        Course(
            id="CS108",
            name="Machine Learning",
            instructor="Dr. Martinez",
            students=["S003", "S006", "S008"],
            priority=4
        )
    ]

    for course in courses:
        scheduler.add_course(course)


    # 2. ADD EXAM SLOTS

    print("\n⏰ Adding Exam Slots...")
    print("-"*40)

    slots = [
        ExamSlot(id="S1", date="2026-07-15", start_time="09:00", end_time="11:00",
                 capacity=30, venue="Hall A"),
        ExamSlot(id="S2", date="2026-07-15", start_time="11:00", end_time="13:00",
                 capacity=30, venue="Hall A"),
        ExamSlot(id="S3", date="2026-07-15", start_time="14:00", end_time="16:00",
                 capacity=30, venue="Hall A"),
        ExamSlot(id="S4", date="2026-07-16", start_time="09:00", end_time="11:00",
                 capacity=25, venue="Hall B"),
        ExamSlot(id="S5", date="2026-07-16", start_time="11:00", end_time="13:00",
                 capacity=25, venue="Hall B"),
        ExamSlot(id="S6", date="2026-07-16", start_time="14:00", end_time="16:00",
                 capacity=25, venue="Hall B"),
        ExamSlot(id="S7", date="2026-07-17", start_time="09:00", end_time="11:00",
                 capacity=20, venue="Hall C"),
        ExamSlot(id="S8", date="2026-07-17", start_time="11:00", end_time="13:00",
                 capacity=20, venue="Hall C"),
    ]

    for slot in slots:
        scheduler.add_slot(slot)


    # 3. ADD CONSTRAINTS

    print("\n🔒 Adding Constraints...")
    print("-"*40)

    scheduler.add_constraint('different_day', ['CS101', 'CS104'])
    scheduler.add_constraint('same_day', ['CS102', 'CS108'])
    scheduler.add_constraint('no_back_to_back', ['CS101', 'CS102'])


    # 4. GENERATE SCHEDULE

    print("\n" + "="*60)
    print("🎯 GENERATING OPTIMAL SCHEDULE...")
    print("="*60)

    schedule = scheduler.solve_csp(use_priority=True)


    # 5. DISPLAY RESULTS

    scheduler.print_schedule()
    scheduler.print_statistics()


    # 6. SAVE AND DOWNLOAD SCHEDULE

    scheduler.save_schedule_json("exam_schedule.json")

    # 7. CONFLICT VERIFICATION

    print("\n✅ CONFLICT VERIFICATION")
    print("-"*40)
    conflicts_found = 0

    for i, sched1 in enumerate(schedule):
        for sched2 in schedule[i+1:]:
            if scheduler.check_student_conflict(sched1.course_id, sched2.course_id):
                slot1 = scheduler.get_slot_by_id(sched1.slot_id)
                slot2 = scheduler.get_slot_by_id(sched2.slot_id)
                if slot1 and slot2 and scheduler.check_time_overlap(slot1, slot2):
                    course1 = next(c for c in courses if c.id == sched1.course_id)
                    course2 = next(c for c in courses if c.id == sched2.course_id)
                    print(f"⚠️ CONFLICT: {course1.name} and {course2.name} share students and overlap!")
                    conflicts_found += 1

    if conflicts_found == 0:
        print("✅ No student conflicts found! Schedule is valid.")
    else:
        print(f"⚠️ Found {conflicts_found} conflicts that need manual resolution.")

    return scheduler


# RUN IN COLAB(DEMO)


if __name__ == "__main__":
    # Run the demo
    scheduler = run_demo()

    print("\n" + "🎉"*20)
    print("AI EXAM SCHEDULER DEMO COMPLETED!")
    print("🎉"*20)

    if IN_COLAB:
        print("\n💡 Next Steps in Colab:")
        print("1. Check the 'exam_schedule.json' file in the file browser (left sidebar)")
        print("2. Modify the courses and slots in the run_demo() function")
        print("3. Run again with your own data")
        print("4. The JSON file was automatically downloaded!")
    else:
        print("\n💡 Next Steps:")
        print("1. Check 'exam_schedule.json' for the exported schedule")
        print("2. Modify the run_demo() function with your own data")
        print("3. Use scheduler.save_schedule_json() to export your schedule")


🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
AI EXAM SCHEDULER - COLAB DEMO
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀


📚 Adding Courses...
----------------------------------------
✅ Added course: Data Structures (ID: CS101, Priority: 5)
✅ Added course: Algorithms (ID: CS102, Priority: 4)
✅ Added course: Database Systems (ID: CS103, Priority: 4)
✅ Added course: Operating Systems (ID: CS104, Priority: 5)
✅ Added course: Computer Networks (ID: CS105, Priority: 3)
✅ Added course: Software Engineering (ID: CS106, Priority: 3)
✅ Added course: Artificial Intelligence (ID: CS107, Priority: 5)
✅ Added course: Machine Learning (ID: CS108, Priority: 4)

⏰ Adding Exam Slots...
----------------------------------------
✅ Added slot: 2026-07-15 09:00-11:00 at Hall A
✅ Added slot: 2026-07-15 11:00-13:00 at Hall A
✅ Added slot: 2026-07-15 14:00-16:00 at Hall A
✅ Added slot: 2026-07-16 09:00-11:00 at Hall B
✅ Added slot: 2026-07-16 11:00-13:00 at Hall B
✅ Added slot: 2026-07-16 14:00-16:00 at Hall B
✅ Added slot: 2026-07-17 09:00-11:00 at Hall C

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 File downloaded automatically!

✅ CONFLICT VERIFICATION
----------------------------------------
✅ No student conflicts found! Schedule is valid.

🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉
AI EXAM SCHEDULER DEMO COMPLETED!
🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉

💡 Next Steps in Colab:
1. Check the 'exam_schedule.json' file in the file browser (left sidebar)
2. Modify the courses and slots in the run_demo() function
3. Run again with your own data
4. The JSON file was automatically downloaded!
